# Anomaly Detection · LSTM Autoencoder · NASA SMAP

## What is an LSTM Autoencoder?
An **autoencoder** is a neural network trained to compress its input into a low-dimensional *bottleneck* representation and then reconstruct the original input from that compressed form.  
We use an **LSTM** (Long Short-Term Memory) as the building block because our input is a *time series* — the LSTM's gating mechanism lets it capture temporal dependencies over hundreds of time-steps without suffering from vanishing gradients.

## Why does it detect anomalies *without labels*?
The model is trained **only on normal (healthy) data**.  It learns a compact representation of "what normal looks like".  At inference time, when it encounters an anomalous pattern it has never seen, it fails to reconstruct it accurately — producing a high **mean-squared error (MSE)**.  We then threshold this reconstruction error to raise an anomaly flag.  This makes the approach *semi-supervised*: we need labels only to tune the threshold and evaluate, not to train.

## NASA SMAP Dataset
The **Soil Moisture Active Passive (SMAP)** dataset, released by NASA JPL and curated by Hundman et al. (NeurIPS 2018), contains:
- **54 telemetry channels** (sensor streams from a satellite)
- **562 labelled anomaly sequences** covering failures such as point anomalies, contextual anomalies, and seasonal shifts
- Pre-split into train (anomaly-free) and test (mixed) splits

We focus on channel **P-1** throughout this notebook, but the pipeline generalises to all 54 channels.

In [ ]:
# Install required packages
!pip install -q torch scikit-learn matplotlib pandas numpy fastapi uvicorn mlflow

In [ ]:
import ast
import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  Device: {device}')

In [ ]:
# ── Download NASA SMAP data ──────────────────────────────────────────────────
import subprocess, shutil

if not os.path.exists('telemanom'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/khundman/telemanom.git'], check=True)
    print('Cloned telemanom repo')
else:
    print('telemanom already present')

DATA_DIR = 'data'
if not os.path.exists(DATA_DIR):
    shutil.copytree('telemanom/data', DATA_DIR)
    print(f'Copied data/ → {DATA_DIR}')
else:
    print(f'{DATA_DIR}/ already present')

# ── Inspect channel P-1 ──────────────────────────────────────────────────────
CHANNEL = 'P-1'
train_raw = np.load(f'{DATA_DIR}/train/{CHANNEL}.npy')
test_raw  = np.load(f'{DATA_DIR}/test/{CHANNEL}.npy')

df_labels = pd.read_csv(f'{DATA_DIR}/labeled_anomalies.csv')
row = df_labels[df_labels['chan_id'] == CHANNEL].iloc[0]
sequences = ast.literal_eval(row['anomaly_sequences'])
label_arr = np.zeros(len(test_raw), dtype=int)
for s, e in sequences:
    label_arr[s:e+1] = 1

print(f'Train : {len(train_raw):,} steps  shape {train_raw.shape}')
print(f'Test  : {len(test_raw):,}  steps  shape {test_raw.shape}')
print(f'Anomaly rate: {label_arr.mean()*100:.2f}%  ({label_arr.sum()} anomalous steps)')

## Sliding Window Concept

Raw telemetry is a 1-D stream of values.  We need to feed the LSTM *fixed-length sub-sequences*.  A **sliding window** of size W with stride 1 extracts every possible contiguous chunk:

```
Signal:  [t0, t1, t2, t3, t4, t5, t6, t7, t8 ...]

Window 0: [t0, t1, t2, t3]      (W=4 shown for clarity)
Window 1:     [t1, t2, t3, t4]
Window 2:         [t2, t3, t4, t5]
Window 3:             [t3, t4, t5, t6]
               ← stride = 1 →
```

**Why stride=1?**  With limited labelled anomaly sequences we want to maximise the number of training samples.  Stride 1 gives the most overlap and therefore the most windows from a fixed-length recording.

At evaluation time each point's anomaly score is the **maximum** reconstruction error among all windows that contain it (*point-adjust*, Hundman et al. 2018).

In [ ]:
# ── Dataset utilities (standalone — no imports from src/) ────────────────────

WINDOW_SIZE = 128
STRIDE = 1

def load_channel(channel, data_dir='data'):
    train_path = os.path.join(data_dir, 'train', f'{channel}.npy')
    test_path  = os.path.join(data_dir, 'test',  f'{channel}.npy')
    train_data = np.load(train_path).astype(np.float32)
    test_data  = np.load(test_path).astype(np.float32)
    if train_data.ndim == 1: train_data = train_data.reshape(-1, 1)
    if test_data.ndim  == 1: test_data  = test_data.reshape(-1, 1)
    scaler = StandardScaler()
    train_data = scaler.fit_transform(train_data)
    test_data  = scaler.transform(test_data)
    return train_data.astype(np.float32), test_data.astype(np.float32), scaler

def load_labels(channel, test_length, data_dir='data'):
    csv_path = os.path.join(data_dir, 'labeled_anomalies.csv')
    df = pd.read_csv(csv_path)
    row = df[df['chan_id'] == channel]
    labels = np.zeros(test_length, dtype=int)
    if row.empty: return labels
    sequences = ast.literal_eval(row['anomaly_sequences'].values[0])
    for s, e in sequences:
        labels[int(s):int(e)+1] = 1
    return labels

def make_windows(data, window_size=WINDOW_SIZE, stride=STRIDE):
    T, F = data.shape
    n_windows = (T - window_size) // stride + 1
    windows = np.lib.stride_tricks.sliding_window_view(data, window_shape=(window_size, F))
    windows = windows[:n_windows:stride].reshape(n_windows, window_size, F)
    return windows.astype(np.float32)

class TimeSeriesDataset(Dataset):
    def __init__(self, windows):
        self.windows = torch.from_numpy(windows)
    def __len__(self): return len(self.windows)
    def __getitem__(self, idx):
        w = self.windows[idx]
        return w, w

def get_dataloaders(train_data, test_data, window_size=WINDOW_SIZE, batch_size=64):
    train_windows = make_windows(train_data, window_size, stride=STRIDE)
    test_windows  = make_windows(test_data,  window_size, stride=1)
    train_ds = TimeSeriesDataset(train_windows)
    test_ds  = TimeSeriesDataset(test_windows)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  drop_last=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)
    return train_loader, test_loader, train_windows, test_windows

# Load and inspect
train_data, test_data, scaler = load_channel(CHANNEL, DATA_DIR)
labels = load_labels(CHANNEL, len(test_data), DATA_DIR)
train_loader, test_loader, train_windows, test_windows = get_dataloaders(
    train_data, test_data, WINDOW_SIZE, batch_size=64
)
print(f'Input dim : {train_data.shape[1]}')
print(f'Train windows : {len(train_windows):,}  |  Test windows : {len(test_windows):,}')

In [ ]:
# ── LSTM Autoencoder (standalone) ────────────────────────────────────────────

class LSTMAutoencoder(nn.Module):
    """
    Input (B,W,F) → Encoder LSTM → Bottleneck h_n[-1] → Decoder LSTM → Linear → Reconstruction (B,W,F)
    Loss = MSE(reconstruction, input)
    """
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.encoder = nn.LSTM(input_dim,   hidden_dim, num_layers, batch_first=True,
                               dropout=dropout if num_layers > 1 else 0.0)
        self.decoder = nn.LSTM(hidden_dim,  hidden_dim, num_layers, batch_first=True,
                               dropout=dropout if num_layers > 1 else 0.0)
        self.out = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        _, (h_n, _) = self.encoder(x)
        bottleneck  = h_n[-1]                                     # (B, hidden_dim)
        dec_in      = bottleneck.unsqueeze(1).repeat(1, x.size(1), 1)  # (B, W, hidden_dim)
        dec_out, _  = self.decoder(dec_in)
        return self.out(dec_out), bottleneck

INPUT_DIM  = train_data.shape[1]
model = LSTMAutoencoder(INPUT_DIM, hidden_dim=64, num_layers=2, dropout=0.2).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────

NUM_EPOCHS  = 50
LR          = 1e-3
WEIGHT_DECAY = 1e-5

criterion  = nn.MSELoss()
optimizer  = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

train_losses, val_losses = [], []
best_val, best_state = float('inf'), None

for epoch in range(1, NUM_EPOCHS + 1):
    # ── train ──
    model.train()
    t_loss = 0.0
    for x, tgt in train_loader:
        x, tgt = x.to(device), tgt.to(device)
        optimizer.zero_grad()
        recon, _ = model(x)
        loss = criterion(recon, tgt)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # prevent exploding gradients
        optimizer.step()
        t_loss += loss.item() * x.size(0)
    t_loss /= len(train_loader.dataset)

    # ── val ──
    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for x, tgt in test_loader:
            x, tgt = x.to(device), tgt.to(device)
            recon, _ = model(x)
            v_loss += criterion(recon, tgt).item() * x.size(0)
    v_loss /= len(test_loader.dataset)

    scheduler.step(v_loss)
    train_losses.append(t_loss)
    val_losses.append(v_loss)

    if v_loss < best_val:
        best_val   = v_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 5 == 0:
        print(f'Ep {epoch:02d} | Train {t_loss:.6f} | Val {v_loss:.6f}')

model.load_state_dict(best_state)
print(f'\nBest val loss: {best_val:.6f}')

## Threshold Sweep & Point-Adjust

### How the threshold sweep works
After training we obtain a reconstruction error (MSE) for every time-step.  These scores are continuous values; we need to convert them into binary *anomaly / normal* predictions.  We sweep `n=200` evenly-spaced candidate thresholds between the minimum and maximum observed score and pick the one that maximises the F1-score on the labelled test set.

### Why point-adjust max-pooling?
Window-level scores must be mapped back to individual time-steps.  Each time-step belongs to `W` overlapping windows.  Rather than averaging, we assign the **maximum** score among all containing windows.  This is conservative: a single strongly anomalous window is enough to flag every time-step it overlaps.  This convention is standard in the SMAP / MSL literature (Hundman et al. 2018) and tends to improve recall without hurting precision significantly.

In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────────

def reconstruction_errors(model, loader, device):
    model.eval()
    errs = []
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            recon, _ = model(x)
            mse = ((recon - x)**2).mean(dim=(1, 2))
            errs.append(mse.cpu())
    return torch.cat(errs).numpy()

def isolation_forest_errors(train_windows, test_windows):
    N_tr, W, F = train_windows.shape
    N_te = test_windows.shape[0]
    Xtr = train_windows.reshape(N_tr, W*F)
    Xte = test_windows.reshape(N_te, W*F)
    clf = IsolationForest(n_estimators=100, contamination='auto', random_state=42, n_jobs=-1)
    clf.fit(Xtr)
    return -clf.score_samples(Xte)

def window_to_point_scores(window_scores, total_length, window_size):
    point_scores = np.zeros(total_length, dtype=np.float32)
    for i, score in enumerate(window_scores):
        end = i + window_size
        if end > total_length: break
        point_scores[i:end] = np.maximum(point_scores[i:end], score)
    return point_scores

def find_best_threshold(scores, labels, n=200):
    thresholds = np.linspace(scores.min(), scores.max(), n)
    best_f1, best_thresh, best_preds = -1., thresholds[0], np.zeros_like(labels)
    for t in thresholds:
        preds = (scores >= t).astype(int)
        f1 = f1_score(labels, preds, average='binary', zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh, best_preds = f1, t, preds
    return best_thresh, best_f1, best_preds

# LSTM-AE
ae_win_scores  = reconstruction_errors(model, test_loader, device)
ae_pt_scores   = window_to_point_scores(ae_win_scores, len(test_data), WINDOW_SIZE)
scored_len     = len(ae_pt_scores)
labels_trim    = labels[:scored_len]

best_thresh, best_f1, best_preds = find_best_threshold(ae_pt_scores, labels_trim)
ae_prec = precision_score(labels_trim, best_preds, zero_division=0)
ae_rec  = recall_score(labels_trim,  best_preds, zero_division=0)
try:
    ae_auc = roc_auc_score(labels_trim, ae_pt_scores)
except ValueError:
    ae_auc = float('nan')

# Isolation Forest baseline
if_win_scores  = isolation_forest_errors(train_windows, test_windows)
if_pt_scores   = window_to_point_scores(if_win_scores, len(test_data), WINDOW_SIZE)
_, if_f1, _    = find_best_threshold(if_pt_scores, labels_trim)
delta          = (best_f1 - if_f1) / max(if_f1, 1e-9) * 100

print('=' * 60)
print(f'LSTM-AE → F1:{best_f1:.3f}  Prec:{ae_prec:.3f}  Rec:{ae_rec:.3f}  AUC:{ae_auc:.3f}')
print(f'Iso.F.  → F1:{if_f1:.3f}')
print(f'Δ F1    → {delta:+.1f}%')
print('=' * 60)

In [ ]:
# ── 3-panel anomaly plot ──────────────────────────────────────────────────────

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
t = np.arange(scored_len)

# Panel 1 — raw signal + true anomaly shading
ax = axes[0]
ax.plot(t, test_data[:scored_len, 0], color='steelblue', lw=0.7, label='Signal')
in_a, start = False, 0
for i in range(len(labels_trim)):
    if labels_trim[i] == 1 and not in_a: start = i; in_a = True
    elif labels_trim[i] == 0 and in_a: ax.axvspan(start, i, color='red', alpha=0.2); in_a = False
if in_a: ax.axvspan(start, len(labels_trim), color='red', alpha=0.2)
ax.legend([ax.lines[0], mpatches.Patch(color='red', alpha=0.3, label='True anomaly')],
          ['Signal', 'True anomaly'], fontsize=8)
ax.set_ylabel('Normalised'); ax.set_title('Raw signal + true anomaly regions')

# Panel 2 — reconstruction error
ax = axes[1]
ax.plot(t, ae_pt_scores, color='darkorange', lw=0.7)
ax.axhline(best_thresh, color='red', ls='--', lw=1.2, label=f'Threshold {best_thresh:.4f}')
ax.legend(fontsize=8); ax.set_ylabel('MSE'); ax.set_title('Point-level reconstruction error')

# Panel 3 — predictions
ax = axes[2]
ax.step(t, labels_trim, color='green', lw=0.8, where='post', label='Ground truth')
ax.step(t, best_preds,  color='red',   lw=0.8, ls='--', where='post', label='Predicted')
ax.set_ylim(-0.05, 1.15); ax.legend(fontsize=8)
ax.set_ylabel('Anomaly'); ax.set_xlabel('Time step'); ax.set_title('Predictions vs ground truth')

plt.tight_layout()
plt.savefig('anomaly_results.png', dpi=150)
plt.show()
print('Saved anomaly_results.png')

In [ ]:
# ── Grad-CAM-style analysis: top-5 highest-error windows ─────────────────────
# For each of the 5 worst-reconstructed windows we plot the original signal
# side-by-side with the model's reconstruction to visualise where it fails.

top5_idx = np.argsort(ae_win_scores)[-5:][::-1]
test_tensor = torch.from_numpy(test_windows).to(device)  # (N, W, F)

model.eval()
fig, axes = plt.subplots(5, 2, figsize=(14, 14))
fig.suptitle('Top-5 highest reconstruction error windows', fontsize=13)

with torch.no_grad():
    for row_idx, win_idx in enumerate(top5_idx):
        x_w = test_tensor[win_idx].unsqueeze(0)       # (1, W, F)
        recon_w, _ = model(x_w)
        original = x_w[0, :, 0].cpu().numpy()
        reconstructed = recon_w[0, :, 0].cpu().numpy()
        err = ae_win_scores[win_idx]

        # Left: original signal
        ax = axes[row_idx, 0]
        ax.plot(original, color='steelblue', lw=1.2)
        ax.set_title(f'Window {win_idx} — original  (MSE={err:.4f})', fontsize=9)
        ax.set_ylabel('Value')

        # Right: reconstruction
        ax = axes[row_idx, 1]
        ax.plot(original,      color='steelblue', lw=0.8, alpha=0.5, label='Original')
        ax.plot(reconstructed, color='red',       lw=1.2, ls='--',   label='Reconstruction')
        ax.legend(fontsize=7)
        ax.set_title(f'Window {win_idx} — reconstruction', fontsize=9)

plt.tight_layout()
plt.savefig('top5_reconstruction.png', dpi=150)
plt.show()
print('Saved top5_reconstruction.png')

In [ ]:
# ── ONNX export ───────────────────────────────────────────────────────────────
import torch.onnx

ONNX_PATH = 'lstm_ae.onnx'
dummy_input = torch.randn(1, WINDOW_SIZE, INPUT_DIM).to(device)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    input_names=['time_series_window'],
    output_names=['reconstruction', 'bottleneck'],
    dynamic_axes={
        'time_series_window': {0: 'batch_size'},
        'reconstruction':     {0: 'batch_size'},
        'bottleneck':         {0: 'batch_size'},
    },
    opset_version=17,
)

size_mb = os.path.getsize(ONNX_PATH) / (1024 ** 2)
print(f'Exported ONNX model → {ONNX_PATH}  ({size_mb:.2f} MB)')